# Download Short-read FQs (From SRA) for isolates sequenced by TB Portals

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
import glob

#### Pandas Viewing Settings

In [3]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [4]:
GitRepo_Data_Dir = "../../Data"

TRUST_IlluminaWGS_Meta_Dir = f"{GitRepo_Data_Dir}/TRUST_IlluminaWGS_Meta"

TRUST_Set1_XLSX = f"{TRUST_IlluminaWGS_Meta_Dir}/2022-05-22_report_Farhat_1st_shipment.v09.xlsx"     



## Parse ShortRead metadata for the T-gen isolates of interest

In [5]:
Repo_DataDir = "../../Data"

TGen_1K_SM_V1_ResultsSummary_Dir = Repo_DataDir + "/211019_SM_TGen_1K_V1_ResultsSummary"

TGen_1K_SampleInfo_TSV_PATH = TGen_1K_SM_V1_ResultsSummary_Dir + "/211019_SM_TGen_1K_SampleInfo_V1.tsv"
TGen_1K_SampleInfo_Filt_TSV_PATH = TGen_1K_SM_V1_ResultsSummary_Dir + "/211019_SM_TGen_1K_SampleInfo_V1.F2andCovFiltered.tsv"

TGen_1K_Info_DF = pd.read_csv(TGen_1K_SampleInfo_Filt_TSV_PATH, sep = "\t")

TGen_1K_RunIDs = list( TGen_1K_Info_DF["RunID"].values )
#VCI_163CI_Names = list( VCI_163CI_Info_DF["SampleName"].values )

ID_To_Lineage_Dict = dict( TGen_1K_Info_DF[["SampleName", "PrimaryLineage_Ill"]].values)

print("# of total samples: ", len(TGen_1K_RunIDs))


# Make sample to lineage mapping dict
#ID_To_PrimLineage_Dict = dict(Hall80CI_AssemblySummary[['SampleID', 'PrimaryLineage_ONTAsm_WiPilonPolish']].values)
#ID_To_Lineage_Dict = dict(Hall80CI_AssemblySummary[['SampleID', 'Lineage_ONTAsm_WiPilonPolish']].values)
#ID_To_Dataset_Dict = dict(Hall80CI_AssemblySummary[['SampleID', 'Dataset_Tag']].values)


# of total samples:  937


In [6]:
TGen_1K_Info_DF.shape


(937, 7)

In [7]:
TGen_1K_Info_DF.head()

,RunID,SampleName,LineageCall_Illumina,F2_Illumina,IlluminaWGSToH37rv_AvrgCov,PrimaryLineage_Ill,Dataset_Tag
0,SRR10397218,SRR10397218,"lineage4,lineage4.2,lineage4.2.1",0.024990,58,lineage4,TBP_TGen_1K
1,SRR10397230,SRR10397230,"lineage4,lineage4.2,lineage4.2.1",0.024843,93,lineage4,TBP_TGen_1K
2,SRR7516345,SRR7516345,"lineage4,lineage4.2,lineage4.2.1",0.023240,54,lineage4,TBP_TGen_1K
3,SRR10379936,SRR10379936,"lineage4,lineage4.1",0.022835,98,lineage4,TBP_TGen_1K
4,SRR7516429,SRR7516429,"lineage4,lineage4.2,lineage4.2.1",0.022706,46,lineage4,TBP_TGen_1K


In [8]:
TGen_1K_Info_DF.query("RunID == 'SRR10379935' ")

,RunID,SampleName,LineageCall_Illumina,F2_Illumina,IlluminaWGSToH37rv_AvrgCov,PrimaryLineage_Ill,Dataset_Tag
65,SRR10379935,SRR10379935,"lineage4,lineage4.1,lineage4.1.2,lineage4.1.2.1",0.01549,49,lineage4,TBP_TGen_1K


## Parse the T-Gen isolates which now have PacBio CCS data

In [9]:
Repo_DataDir = "../../Data"

TBP_PB_CCS_MetaDir = f"{Repo_DataDir}/221017_TBPortals_LRandSR_InputDataTSVs"

TBP_PB_20CI_Set1_TSV = f"{TBP_PB_CCS_MetaDir}/221017.TBPortals.PBCCS.SampleIDs.20CI.Set1.tsv"

In [10]:
!ls -alh $TBP_PB_CCS_MetaDir

total 160K
drwxr-sr-x  2 mm774 farhat  147 Oct 18 00:31 .
drwxr-sr-x 38 mm774 farhat 1.9K Oct 17 16:16 ..
-rw-r--r--  1 mm774 farhat 1.7K Oct 17 16:24 221017.TBPortals.PBCCS.SampleIDs.20CI.Set1.tsv
-rw-r--r--  1 mm774 farhat 8.8K Oct 18 00:31 221018.TBPortals.2022.MtbWGS.LRandSR.Set1.20CI.InputWGS.PATHs.tsv


In [11]:
TBP_CCS_S1_SampleInfo_DF = pd.read_csv(TBP_PB_20CI_Set1_TSV, sep = "\t")
TBP_CCS_S1_SampleInfo_DF.shape

(20, 9)

In [12]:
TBP_CCS_S1_SampleInfo_DF.head()

,Sample_ID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note
0,DNA0428,SRR10379945,PutativeRecomb,Event_001,"vapC25,vapB25",Within_Event,Y,P7529,NaN
1,DNA373,SRR10397263,PutativeRecomb,Event_001,"vapC25,vapB25",Outside_Event,Y,P7529,NaN
2,DNA022,SRR6807674,PutativeRecomb,Event_002,Rv0393,Within_Event,Y,P7544,NaN
3,DNA146,SRR10380093,PutativeRecomb,Event_002 | Event_024,"Rv0393 | PPE59,Rv3430c",Outside_Event | Within_Event,Y,P7544,NaN
4,DNA0530,SRR10380218,PutativeRecomb,Event_003,"vapB31,vapC31",Outside_Event,Y,P7544,NaN


In [13]:
!module load sratoolkit/2.10.7

In [14]:
!fasterq-dump --help


Usage: fasterq-dump [ options ] [ accessions(s)... ]

Parameters:

  accessions(s)                    list of accessions to process


Options:

  -o|--outfile <path>              full path of outputfile (overrides usage
                                     of current directory and given accession)
  -O|--outdir <path>               path for outputfile (overrides usage of
                                     current directory, but uses given
                                     accession)
  -b|--bufsize <size>              size of file-buffer (dflt=1MB, takes
                                     number or number and unit where unit is
                                     one of (K|M|G) case-insensitive)
  -c|--curcache <size>             size of cursor-cache (dflt=10MB, takes
                                     number or number and unit where unit is
                                     one of (K|M|G) case-insensitive)
  -m|--mem <size>                  memory limit for sorting (dflt=

## Define directory to download data to on O2 (in personal storage directory)

In [15]:
mm774_FarhatDir = "/n/data1/hms/dbmi/farhat/mm774"
DownloadedData_Dir = mm774_FarhatDir + "/DownloadedData"

In [16]:
ProjectID = "221017_TBPortals_SR_Tgen_IsolatesOfInterest"

Project_DownloadedData_Dir = DownloadedData_Dir + "/" + ProjectID

!mkdir -p $Project_DownloadedData_Dir

In [17]:
!ls -lah $DownloadedData_Dir

total 266G
drwxrwsr-x  21 mm774 farhat  880 Oct 17 16:59 .
drwxrwsr-x  12 mm774 farhat  390 Sep 28 15:00 ..
drwxrwsr-x  26 mm774 farhat  673 Feb 22  2021 201123_TB_Portals_15CI_SelectedIsolates_V1
drwxr-sr-x   4 mm774 farhat   74 Oct 12 10:54 2022_TB_Portals_PB_Data
drwxrwsr-x  34 mm774 farhat  898 Apr 14  2021 210414_TB_Portals_32CI_All_SelectedIsolates
drwxrwsr-x  10 mm774 farhat  225 Apr 14  2021 210414_TB_Portals_8CI_SelectedIsolates_Part3
drwxrwsr-x  26 mm774 farhat  570 Jan 10  2022 220110_Peker2021_ONT
drwxrwsr-x 153 mm774 farhat 3.9K Oct 17 16:55 220309_Hall2022_LRandSR
drwxrwsr-x   2 mm774 farhat    0 Oct 17 16:59 221017_TBPortals_SR_Tgen_IsolatesOfInterest
drwxrwsr-x  22 mm774 farhat  486 Oct 17 16:59 221017_TBPortals_Tgen_Selected_SR
drwxrwsr-x  42 mm774 farhat 1.1K Sep 13  2019 ChinerOms_IllData_PRJEB31443_And_PRJEB27802
-rw-r--r--   1 mm774 farhat  14K Oct  5 11:59 DATA_P7529_20221003_info.txt
drwxrwsr-x  34 mm774 farhat 1.3K Apr 15  2021 ForMax_NIAID_LRS
drwxr-sr-x   2 mm

In [18]:
ProjectID = "221017_TBPortals_Tgen_Selected_SR"

In [19]:
mm774_FarhatDir = "/n/data1/hms/dbmi/farhat/mm774"
DownloadedData_Dir = mm774_FarhatDir + "/DownloadedData"
Project_DownloadedData_Dir = DownloadedData_Dir + "/" + ProjectID

!mkdir -p $Project_DownloadedData_Dir

In [20]:
TBP_CCS_S1_SampleInfo_DF.head(1)

,Sample_ID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note
0,DNA0428,SRR10379945,PutativeRecomb,Event_001,"vapC25,vapB25",Within_Event,Y,P7529,NaN


In [21]:
for idx, row in TBP_CCS_S1_SampleInfo_DF.iterrows():
    
    i_SR_RunAcc = row["SRA_RunAcc_SR"]
    i_SampleID = row["Sample_ID"]

    print(idx, i_SR_RunAcc, i_SampleID)
    
    i_Sample_Dir = f"{Project_DownloadedData_Dir}/{i_SampleID}"
    !mkdir $i_Sample_Dir
 
    #!fasterq-dump prefetch $i_SR_RunAcc
    
    download_Command_FQdump = f"time fastq-dump --gzip --split-files --outdir {i_Sample_Dir}/ {i_SR_RunAcc} "
    print(download_Command_FQdump)

    !sbatch -p short -c 1 -t 0-1:45 --mem=1G --mail-user=mgmarin@g.harvard.edu --wrap="$download_Command_FQdump"       

    
    #!time fastq-dump --split-files --outdir $i_Sample_Dir/ $i_SR_RunAcc
    
    print("-----------------------------------------")
    #break 


0 SRR10379945 DNA0428
time fastq-dump --gzip --split-files --outdir /n/data1/hms/dbmi/farhat/mm774/DownloadedData/221017_TBPortals_Tgen_Selected_SR/DNA0428/ SRR10379945 
Submitted batch job 63642155
-----------------------------------------
1 SRR10397263 DNA373
time fastq-dump --gzip --split-files --outdir /n/data1/hms/dbmi/farhat/mm774/DownloadedData/221017_TBPortals_Tgen_Selected_SR/DNA373/ SRR10397263 
Submitted batch job 63642156
-----------------------------------------
2 SRR6807674 DNA022
time fastq-dump --gzip --split-files --outdir /n/data1/hms/dbmi/farhat/mm774/DownloadedData/221017_TBPortals_Tgen_Selected_SR/DNA022/ SRR6807674 
Submitted batch job 63642157
-----------------------------------------
3 SRR10380093 DNA146
time fastq-dump --gzip --split-files --outdir /n/data1/hms/dbmi/farhat/mm774/DownloadedData/221017_TBPortals_Tgen_Selected_SR/DNA146/ SRR10380093 
Submitted batch job 63642158
-----------------------------------------
4 SRR10380218 DNA0530
time fastq-dump --gzip

In [22]:
#!squeue -u mm774

In [23]:
!echo $i_Sample_Dir

/n/data1/hms/dbmi/farhat/mm774/DownloadedData/221017_TBPortals_Tgen_Selected_SR/DNA435


In [24]:
!ls -lah $i_Sample_Dir

total 64K
drwxr-sr-x  2 mm774 farhat   0 Oct 18 00:44 .
drwxrwsr-x 22 mm774 farhat 486 Oct 18 00:44 ..


In [25]:
#!du -sh $Project_DownloadedData_Dir/*

#### Next Step 1: Creata new notebook for defining the PATHs to both the PacBio CCS read FASTQs + the Illumina WGS FASTQs

#### Next Step 2: Run the PB-CCS Asm pipeline on the LR+SR Data